# 04 · SciSpaCy NER on Medical Conversations (Kaggle)

Extracts biomedical named entities (diseases, chemicals/drugs) from the `utterance` column  
using **`en_ner_bc5cdr_md`** — a SciSpaCy model trained on the BC5CDR corpus, covering:
- `DISEASE` — diseases, disorders, syndromes, symptoms  
- `CHEMICAL` — drugs, chemicals, compounds  

Processing uses `nlp.pipe()` for efficient batch inference.

---
**Kaggle setup notes:**
- Upload `integrated_conversations.csv` as a Kaggle Dataset and attach it to this notebook.
- Input files are available at `/kaggle/input/<dataset-slug>/`.
- Output files should be written to `/kaggle/working/`.
- Internet must be **enabled** in notebook settings (Settings → Internet → On) for the `pip install` cells to work.
- GPU is not required; CPU is sufficient for this NER pipeline.

## 0 · Environment detection

In [ ]:
import os
from pathlib import Path

# Detect Kaggle vs local environment
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # ── Kaggle paths ──────────────────────────────────────────────────────────
    # Update the dataset slug below to match your attached dataset name.
    # Example: if you uploaded a dataset called "medical-conversations",
    # the path will be /kaggle/input/medical-conversations/
    DATASET_SLUG = 'your-dataset-slug'  # <-- UPDATE THIS
    INPUT_CSV  = Path(f'/kaggle/input/{DATASET_SLUG}/integrated_conversations.csv')
    OUTPUT_CSV = Path('/kaggle/working/integrated_conversations_ner.csv')
else:
    # ── Local paths ───────────────────────────────────────────────────────────
    DATA_DIR   = Path('../data/processed')
    INPUT_CSV  = DATA_DIR / 'integrated_conversations.csv'
    OUTPUT_CSV = DATA_DIR / 'integrated_conversations_ner.csv'

print(f'Running on : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Input  : {INPUT_CSV}')
print(f'Output : {OUTPUT_CSV}')

## 1 · Install dependencies

> **Kaggle:** Make sure **Internet is enabled** in notebook settings before running this cell.  
> The `en_ner_bc5cdr_md` model (~120 MB) is downloaded directly from the Allen AI S3 bucket.

In [ ]:
# Install scispacy and the BC5CDR NER model
# -q suppresses verbose pip output on Kaggle
!pip install -q scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

## 2 · Imports & configuration

In [ ]:
import pandas as pd
import spacy
from tqdm.auto import tqdm

# ── NER settings ──────────────────────────────────────────────────────────────
MODEL_NAME = 'en_ner_bc5cdr_md'
BATCH_SIZE = 256   # number of texts per batch; lower if Kaggle RAM is tight
N_PROCESS  = 1     # keep at 1 on Kaggle (multiprocessing is restricted)

print(f'Model  : {MODEL_NAME}')
print(f'Input  : {INPUT_CSV}')
print(f'Output : {OUTPUT_CSV}')

## 3 · Load dataset

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Rows   : {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Sanity-check the utterance column ─────────────────────────────────────────
missing = df['utterance'].isna().sum()
print(f'Missing utterances : {missing}')

# Fill NaN with empty string so nlp.pipe() never receives None
df['utterance'] = df['utterance'].fillna('').astype(str)
print(f'Sample utterance   : {df["utterance"].iloc[2][:120]}')

## 4 · Load SciSpaCy model

In [ ]:
nlp = spacy.load(MODEL_NAME)

# Disable unused pipeline components to speed up inference
for pipe in ['parser', 'lemmatizer']:
    if pipe in nlp.pipe_names:
        nlp.disable_pipe(pipe)

print(f'Loaded  : {MODEL_NAME}')
print(f'Pipeline: {nlp.pipe_names}')
print(f'Entities: {nlp.get_pipe("ner").labels}')

## 5 · Batch NER extraction

`nlp.pipe()` streams utterances in batches — far faster than calling `nlp(text)` row-by-row.

In [ ]:
def extract_entities(docs):
    """
    Given an iterable of spaCy Doc objects, return two parallel lists:
      - entity_texts  : list of entity surface strings per doc
      - entity_labels : list of entity label strings per doc
    """
    entity_texts  = []
    entity_labels = []
    for doc in docs:
        texts  = [ent.text   for ent in doc.ents]
        labels = [ent.label_ for ent in doc.ents]
        entity_texts.append(texts)
        entity_labels.append(labels)
    return entity_texts, entity_labels


# ── Run nlp.pipe() over all utterances ────────────────────────────────────────
utterances = df['utterance'].tolist()
print(f'Processing {len(utterances):,} utterances  '
      f'(batch_size={BATCH_SIZE}) …')

docs = nlp.pipe(
    utterances,
    batch_size=BATCH_SIZE,
    n_process=N_PROCESS
)

docs_with_progress = tqdm(docs, total=len(utterances), desc='NER')
entity_texts, entity_labels = extract_entities(docs_with_progress)
print('Done.')

## 6 · Attach results to the dataframe

In [ ]:
# Store as lists (native Python objects in each cell)
df['extracted_entities'] = entity_texts
df['entity_labels']      = entity_labels

# Quick stats
has_entity = df['extracted_entities'].apply(lambda x: len(x) > 0)
total_ents = df['extracted_entities'].apply(len).sum()

print(f'Utterances with ≥1 entity : {has_entity.sum():,} '
      f'({has_entity.mean()*100:.1f}%)')
print(f'Total entities extracted  : {total_ents:,}')
df[['utterance', 'extracted_entities', 'entity_labels']].head(10)

## 7 · Entity frequency breakdown

In [ ]:
from collections import Counter

# Flatten all (entity, label) pairs for counting
all_pairs = [
    (ent, lbl)
    for ents, lbls in zip(entity_texts, entity_labels)
    for ent, lbl in zip(ents, lbls)
]

# Label distribution
label_counts = Counter(lbl for _, lbl in all_pairs)
print('Entity label distribution:')
for label, count in label_counts.most_common():
    print(f'  {label:<12} {count:>7,}')

print()

# Top 20 most frequent entities
top_entities = Counter(ent.lower() for ent, _ in all_pairs).most_common(20)
print('Top 20 entities:')
for ent, cnt in top_entities:
    print(f'  {cnt:>6,}  {ent}')

## 8 · Save output CSV

> **Kaggle:** The file will be saved to `/kaggle/working/` and will appear in the  
> **Output** tab of this notebook after the run completes. You can then download it  
> or use it as an input dataset for a downstream notebook.

In [ ]:
# Convert list columns to pipe-delimited strings for clean CSV storage
df_out = df.copy()
df_out['extracted_entities'] = df_out['extracted_entities'].apply(
    lambda x: ' | '.join(x) if x else ''
)
df_out['entity_labels'] = df_out['entity_labels'].apply(
    lambda x: ' | '.join(x) if x else ''
)

df_out.to_csv(OUTPUT_CSV, index=False)
print(f'✅  Saved {len(df_out):,} rows → {OUTPUT_CSV}')
df_out[['utterance', 'extracted_entities', 'entity_labels']].head(5)

---
### Summary

| Step | Detail |
|------|--------|
| Model | `en_ner_bc5cdr_md` (SciSpaCy, BC5CDR corpus) |
| Entity types | `DISEASE`, `CHEMICAL` |
| Processing | `nlp.pipe()` with batch_size=256 |
| Output columns | `extracted_entities` (pipe-delimited text), `entity_labels` (pipe-delimited labels) |
| Output file | `integrated_conversations_ner.csv` |

---
### Kaggle checklist

- [ ] Dataset attached and `DATASET_SLUG` updated in Cell 0
- [ ] **Internet enabled** in notebook settings
- [ ] Run All → output CSV appears in the **Output** tab